# Satellite-start map: end of the period-doubling bubble, and chaos beyond it (22/10/2026)

Two questions on `heatmap_lyapunov_satellite_start_*` (`lyapunov_heatmap_satellite_start.ipynb`), $\kappa_b = \pi/10.4$ (code convention), $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$:
1. Draw the **end of the bubble**, i.e. the reverse period doubling where the satellite becomes stable again, and compare it with the red → blue transition of the map.
2. The map of 12/10/2026 is chaotic almost everywhere above the bubble, the map of 22/10/2026 is not. **Where does that chaos come from?**

The two maps differ in two ways: the initial state (analytic satellite of App. "Satellites", `satellite_hb`, in 12_10; exact HB satellite, `Hill_satellite_seed_grid.ipynb`, in 22_10)
and the transient discarded before $\lambda$ is measured ($T_\mathrm{tr} = 15/\kappa_b$ in 12_10, as in the notebook snapshot of 12/10; $300/\kappa_b$ in 22_10). Section 3 separates the two.

Harmonic balance and Floquet multipliers: `catchaos` (`ats.HB`, `hill.monodromy`, `hill.memory_phase`; first seed `seeds.satellite_hb`, `seeds.seed_z`). Time integrations: `satellite_start_beyond_bubble.jl` (Julia, same equations).

In [ ]:
import json, os, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import *                      # w_a, phi_a, phi_b, E_J, kappa_b (code convention), G0, MHz, g2_max
from catchaos.plotting import *                    # Plot_setting (figsize, figsize_car, ...)
from catchaos.ats import HB
from catchaos.hill import monodromy, memory_phase
from catchaos.seeds import satellite_hb, seed_z

HERE = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/studies/torus_crisis/"
OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/"

r0 = 10.12
K_REG = {"A": 1.0, "B": 8.0, "C": 11.0}                          # kappa_b / kappa_b0 of the three examples
REG_NAME = {"A": "below the cusp, period doubling", "B": "below the cusp, no period doubling", "C": "beyond the cusp"}

# 1. Edges of the period-doubling bubble vs $\varepsilon_d/g_2$
For each $\varepsilon_d/g_2$ (steps of 0.25), the satellite at the right edge of the map (122.5 MHz, $\varepsilon_p = 1.1$) is continued in $\varepsilon_d/g_2$ from 10.12, then down in $g_2$ (steps of 0.25 MHz) along the satellite.
The most negative real multiplier crosses $-1$ twice: first at the **upper edge** (reverse period doubling, end of the bubble), then at the **lower edge** (the period doubling of `torus_breakdown`).
Crossings are interpolated linearly. Below $\varepsilon_d/g_2 \simeq 4.25$ the upper edge is beyond the map (above 122.5 MHz).

In [ ]:
N_H, r0 = 16, 10.12
hb = HB(N_H, r=r0)
s60 = 60/g2_max
hb.set_s(s60)
z = hb.solve(seed_z(satellite_hb(s60, r0), N_H))
G_RIGHT = 122.5                                   # right edge of the map (eps_p = 1.1)
for s in np.linspace(s60, G_RIGHT/g2_max, 40)[1:]:
    hb.set_s(s); z = hb.solve(z)

In [ ]:
def most_negative_real(hb, z):
    mu = monodromy(hb, z)
    re = mu[np.abs(mu.imag) < 1e-9].real
    return re.min() if len(re) else 0.0

def bubble_edges(z_top, r, dg=0.25, g_min=45.0):
    # down in g2 along the satellite from G_RIGHT: g2 where the most negative real multiplier crosses -1
    z, g, prev, edges = z_top.copy(), G_RIGHT, None, []
    while g > g_min and len(edges) < 2:
        hb.set_s((g - dg)/g2_max, r)
        try:
            zn = hb.solve(z)
        except RuntimeError:
            break
        if abs(memory_phase(zn) - memory_phase(z)) > 0.1:
            break                                     # left the satellite (fold)
        m = most_negative_real(hb, zn)
        if prev is not None and (prev + 1)*(m + 1) < 0:
            edges.append(g - dg*(-1 - prev)/(m - prev))
        prev, z, g = m, zn, g - dg
    return edges

In [ ]:
RATIOS = np.round(np.arange(2.0, 13.51, 0.25), 2)
upper, lower = {}, {}
for part in (RATIOS[RATIOS >= r0], RATIOS[RATIOS < r0][::-1]):
    zr, r_prev = z.copy(), r0
    for r in part:
        for rr in np.linspace(r_prev, r, 6)[1:]:
            hb.set_s(G_RIGHT/g2_max, rr); zr = hb.solve(zr)
        r_prev = r
        e = bubble_edges(zr, r)
        upper[r], lower[r] = (e[0], e[1]) if len(e) == 2 else (np.nan, e[0] if e else np.nan)
up_line = np.array([upper[r] for r in RATIOS]); lo_line = np.array([lower[r] for r in RATIOS])
for r in RATIOS[::4]:
    print(f"eps_d/g2 = {r:5.2f}: bubble {lower[r]:6.2f} - {upper[r]:6.2f} MHz")
json.dump({"description": "Period-doubling bubble of the satellite (kappa_b = pi/10.4, code): lower edge = period doubling, upper edge = reverse period doubling "
                          "(end of the bubble). NaN: beyond the map (> 122.5 MHz). From satellite_start_bubble_edge.ipynb",
           "eps_d_over_g2": RATIOS.tolist(), "g2_lower_MHz": [None if np.isnan(v) else v for v in lo_line],
           "g2_upper_MHz": [None if np.isnan(v) else v for v in up_line]},
          open(OUT_DIR + "torus_crisis/satellite_pd_bubble_edges_22_10_2026.json", "w"))

# 2. The two maps with the edges of the bubble
Rows: map of 22/10 (exact seed, $T_\mathrm{tr} = 300/\kappa_b$) and of 12/10 (analytic seed, $T_\mathrm{tr} = 15/\kappa_b$). Columns: sign of $\lambda$, end state (satellite, cat, chaotic), from the saved JSON files.
Lines: satellite fold (dashed black), Neimark–Sacker of the period-2 orbit (solid black, `Hill_method_doubling_corrected`), lower edge of the bubble (dotted gold), **end of the bubble** (solid gold).

In [ ]:
from matplotlib.colors import ListedColormap
def load_map(tag):
    folder = "lyapunov_maps/" if tag == "22_10" else "../archive/data/"          # the 12/10 map is superseded (archive)
    to_arr = lambda name: np.array([[np.nan if v is None else v for v in col]
                                    for col in json.load(open(OUT_DIR + folder + f"{name}_satellite_start_{tag}_2026.json"))], float).T   # (eps_d/g2, eps_p)
    return to_arr("data_lyapunov"), to_arr("end_state")
g2_map = np.array(json.load(open(OUT_DIR + "lyapunov_maps/g2_list_satellite_start_22_10_2026.json")))/2/np.pi*1e3
a2_map = np.array(json.load(open(OUT_DIR + "lyapunov_maps/alpha_sq_list_satellite_start_22_10_2026.json")))
SEED = json.load(open(OUT_DIR + "hill_orbits/satellite_seed_hill_22_10_2026.json"))
g_fold = np.array(SEED["g2_fold_MHz"], float); r_fold = np.array(SEED["eps_d_over_g2"], float)
g_ns = np.array(json.load(open(OUT_DIR + "hill_orbits/g2_list_12_10_2026_Hill_unstable.json")), float)
r_ns = np.array(json.load(open(OUT_DIR + "hill_orbits/eps_d_ratio_list_12_10_2026_Hill_unstable.json")), float)
MAPS = {tag: load_map(tag) for tag in ("22_10", "12_10")}

def lines(ax):
    ax.plot(g_fold, r_fold/2, "k--", lw=0.8)
    ax.plot(g_ns, r_ns/2, "k-", lw=1.0)
    ax.plot(lo_line, RATIOS/2, ":", color="gold", lw=1.5)
    ax.plot(up_line, RATIOS/2, "-", color="gold", lw=1.5)
    ax.set_xlim(g2_map[0], g2_map[-1]); ax.set_ylim(a2_map[0], a2_map[-1])
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")

In [ ]:
END_COLORS = [(13/256, 76/256, 116/256), (134/256, 187/256, 216/256), (185/256, 21/256, 21/256)]   # satellite, cat, chaotic
fig, axes = plt.subplots(2, 2, figsize=(2.2*figsize[0], 2.0*figsize[1]))
for row, (tag, title) in enumerate((("22_10", r"22/10: exact seed, $T_\mathrm{tr} = 300/\kappa_b$"), ("12_10", r"12/10: analytic seed, $T_\mathrm{tr} = 15/\kappa_b$"))):
    L, Es = MAPS[tag]
    axes[row, 0].pcolormesh(g2_map, a2_map, np.sign(L), cmap=ListedColormap(["navy", "darkred"]), vmin=-1, vmax=1, shading="nearest", rasterized=True)
    axes[row, 1].pcolormesh(g2_map, a2_map, Es, cmap=ListedColormap(END_COLORS), vmin=0.5, vmax=3.5, shading="nearest", rasterized=True)
    for ax, name in zip(axes[row], ("sign of the Lyapunov exponent", "end state")):
        lines(ax); ax.set_title(f"{title}: {name}", fontsize=6)
handles = [mpl.lines.Line2D([], [], color="k", ls="--", lw=0.8), mpl.lines.Line2D([], [], color="k", lw=1.0),
           mpl.lines.Line2D([], [], color="gold", ls=":", lw=1.5), mpl.lines.Line2D([], [], color="gold", lw=1.5)] + \
          [mpl.patches.Patch(color=c) for c in END_COLORS]
labels = ["satellite fold", "Neimark–Sacker (period 2)", "period doubling (lower edge of the bubble)", "end of the bubble (reverse period doubling)",
          "satellite", "cat", "chaotic"]
fig.legend(handles, labels, fontsize=5, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR + "heatmap_lyapunov_satellite_start_bubble_22_10_2026.pdf", bbox_inches="tight")
plt.show()

In [ ]:
print("Main chaotic band of each row (from its first chaotic point to the next non-chaotic one) vs the end of the bubble:")
for tag in ("22_10", "12_10"):
    L = MAPS[tag][0]
    print(f"  map {tag}")
    for r in (13.5, 12.0, 10.0, 8.0, 6.0, 4.5):
        j = np.argmin(np.abs(2*a2_map - r)); row = L[j]
        i0 = np.argmax(row > 0); i1 = i0 + np.argmax(row[i0:] < 0)
        e = upper[r]
        above = row[g2_map > e]
        print(f"    eps_d/g2 = {r:5.2f}: band {g2_map[i0]:6.1f} -> {g2_map[i1]:6.1f} MHz, end of the bubble {e:6.1f} MHz, "
              f"chaotic points above it: {np.nansum(above > 0):.0f}/{np.sum(np.isfinite(above))}")

# 3. Chaos above the bubble in the 12/10 map: initial state or transient?
Every grid point of the maps above the end of the bubble (+0.5 MHz) with $\varepsilon_d/g_2 = 4.5, 5, \dots, 13.5$ (where the end of the bubble is inside the map). At each point, `satellite_start_beyond_bubble.jl` computes:
- $\lambda_1$ from the **analytic seed** with $T_\mathrm{tr} = 15/\kappa_b$ (protocol of 12/10) and $300/\kappa_b$ (protocol of 22/10), and from the **exact satellite** with $T_\mathrm{tr} = 15/\kappa_b$;
- the fate of the analytic-seed trajectory over $2000/\kappa_b$: time at which it lands on the satellite pair (stroboscopic distance below $10^{-3}$), or $\lambda_1$ over the last $100/\kappa_b$;
- the basin along the seed direction: the exact satellite kicked by $\epsilon\,(y_\mathrm{seed} - y_\mathrm{sat})/|y_\mathrm{seed} - y_\mathrm{sat}|$, $\epsilon = 0.003, 0.01, 0.03, 0.1$, back on it after $300/\kappa_b$ or not.

Distances are in the $x^\lambda$ units of the harmonic balance ($x_a = \sqrt2\varphi_a\mathrm{Re}\,a$, ...). The Julia run takes about 3 minutes with 8 threads; it is skipped if its output exists and `RUN_JULIA = False`.

In [ ]:
S_state, S_eps, S_rat = SEED["state"], np.array(SEED["eps_p_list"]), np.array(SEED["eps_d_over_g2"])
L12 = MAPS["12_10"][0]
pts = []
for r in np.arange(4.5, 13.51, 0.5):
    if np.isnan(upper[r]):
        continue
    j = int(np.argmin(np.abs(S_rat - r)))
    for i, ep in enumerate(S_eps):
        g = MHz(np.sin(ep)*G0)
        A = satellite_hb(np.sin(ep), S_rat[j])
        if g < upper[r] + 0.5 or S_state[j][i] is None or A is None:
            continue
        a = A["a0"] + A["a1"] + A["am1"]; b = A["b0"] + A["bm1"] + A["b2"]
        pts.append(dict(j=j, i=i, ratio=float(S_rat[j]), s=float(np.sin(ep)), g2_MHz=float(g), y_exact=[float(v) for v in S_state[j][i]],
                        y_analytic=[float(np.sqrt(2)*a.real), float(np.sqrt(2)*b.real), float(np.sqrt(2)*a.imag), float(2*np.sqrt(2)*b.imag)],
                        lam_map_12_10=None if np.isnan(L12[j, i]) else float(L12[j, i])))
json.dump(pts, open(OUT_DIR + "torus_crisis/satellite_start_beyond_bubble_points_22_10_2026.json", "w"))
print(f"{len(pts)} points above the end of the bubble")

RUN_JULIA = False
OUT = OUT_DIR + "torus_crisis/satellite_start_beyond_bubble_22_10_2026.json"
if RUN_JULIA or not os.path.exists(OUT):
    run = subprocess.run(["julia", "-t", "8", "--project=/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia", HERE + "satellite_start_beyond_bubble.jl"],
                         capture_output=True, text=True)
    print(run.stdout[-2000:], run.stderr[-2000:])
RES = json.load(open(OUT)); P = RES["points"]; eps_kick = RES["eps_kick"]
n = len(P)
lam15, lam300, lam15x = (np.array([p[k] for p in P]) for k in ("lam15", "lam300", "lam15_exact"))
landed = np.array([p["t_land_over_kappa"] is not None for p in P]); lam_end = np.array([p["lam_end"] for p in P])
d_seed = np.array([p["dist_seed"] for p in P]); kicks = np.array([p["kick_returns"] for p in P])
print(f"distance analytic seed - exact satellite: median {np.median(d_seed):.3f} (range {d_seed.min():.3f}-{d_seed.max():.3f})")
print(f"lambda_1 > 0: map 12_10 {sum(p['lam_map_12_10'] > 0 for p in pts if p['lam_map_12_10'] is not None)}/{n}; "
      f"analytic seed, Ttr = 15/kappa {np.sum(lam15 > 0)}/{n}; Ttr = 300/kappa {np.sum(lam300 > 0)}/{n}; exact seed, Ttr = 15/kappa {np.sum(lam15x > 0)}/{n}")
print(f"analytic seed after 2000/kappa: on the satellite {landed.sum()}/{n}; still chaotic (lambda_1 > 0.01 kappa over the last 100/kappa) {np.sum(~landed & (lam_end > 0.01))}/{n}; "
      f"on another regular attractor {np.sum(~landed & (lam_end <= 0.01))}/{n}")
for e, frac in zip(eps_kick, kicks.mean(axis=0)):
    print(f"kick of {e:5.3f} from the exact satellite towards the seed: back on the satellite for {100*frac:5.1f} % of the points")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2.2*figsize[0], figsize[1]))
L = MAPS["12_10"][0]
ax = axes[0]
ax.pcolormesh(g2_map, a2_map, np.sign(L), cmap=ListedColormap(["navy", "darkred"]), vmin=-1, vmax=1, shading="nearest", rasterized=True, alpha=0.25)
cls = np.where(landed, 0, np.where(lam_end > 0.01, 3, np.where(lam300 > 0, 2, np.where(lam15 > 0, 1, 4))))
CLS = [("lands on the satellite", "white"), (r"chaotic for $< 300/\kappa_b$, then regular (not the satellite)", "orange"),
       (r"chaotic for $> 300/\kappa_b$, then regular (not the satellite)", "red"), (r"still chaotic at $2000/\kappa_b$", "darkred"),
       ("never chaotic, other attractor", "steelblue")]
for c, (lab, col) in enumerate(CLS):
    m = cls == c
    ax.scatter(np.array([p["g2_MHz"] for p in P])[m], np.array([p["ratio"] for p in P])[m]/2, s=6, color=col, edgecolor="k", lw=0.2, label=f"{lab} ({m.sum()})")
lines(ax); ax.set_title("12/10 map (faded) and fate of the analytic-seed trajectories", fontsize=6)
ax.legend(fontsize=4.5, loc="lower left")
ax = axes[1]
ax.semilogx(eps_kick, 100*kicks.mean(axis=0), "ko-", ms=3)
ax.axvline(np.median(d_seed), color="gray", ls="--", lw=0.8, label="median distance of the analytic seed")
ax.set_xlabel(r"kick $\epsilon$ ($x^\lambda$ units)"); ax.set_ylabel("back on the satellite (%)"); ax.legend(fontsize=5)
ax.set_title("basin of the satellite along the seed direction", fontsize=6)
plt.tight_layout()
plt.savefig(FIG_DIR + "satellite_start_beyond_bubble_22_10_2026.pdf", bbox_inches="tight")
plt.show()

# 4. Is the chaos beyond the bubble the remnant of the bubble's chaotic attractor?
**Already known** (`transient_chaos_shannon_entropy.ipynb`, $\varepsilon_d = 5.2\,g_2$, i.e. $\varepsilon_d/2g_2 = 2.6$, random initial memory states): the chaotic attractor survives up to $g_c \simeq 87.6$ MHz ($\varepsilon_p = 0.695$);
above it the chaos is transient, with lifetimes diverging at $g_c$ as $(\varepsilon_p - \varepsilon_c)^{-1.7}$, the signature of a **boundary crisis**: the attractor becomes a chaotic saddle. At that cat size the bubble is 80.5–116.4 MHz, so the crisis is well inside it
(as on the 22/10 map, where the band ends near 90 MHz for small cats). That scan stops at 122 MHz and does not test the region beyond the bubble.

**Here**, at $\varepsilon_d = 10\,g_2$ (a row of the map grid; bubble 75.4–103.9 MHz), `bubble_crisis_lifetime.jl` runs two families of initial states at each $g_2$:
- **attractor**: 40 points of the bubble's chaotic attractor taken at $g_0 = 100$ MHz;
- **seed**: the analytic satellite at that $g_2$ (the initial state of the 12/10 map), with 40 random kicks of $10^{-3}$.

Escape = the trajectory has settled on a periodic orbit (period $\le 8$ on the section at $T_a$); escape time = first iterate within $10^{-3}$ of it; censored at $20\,000\,T_a \approx 1490/\kappa_b$.
**Test**: if the chaos met from the analytic seed beyond the bubble is the remnant (chaotic saddle) of the bubble's attractor, both families have the same escape rate $1/\tau$ (a saddle forgets where the trajectory came from) and visit the same region of the section,
and $\tau(g_2)$ of the attractor family diverges at the crisis point.
Times below are in units of $1/\kappa_b$ in the code convention (double them for the article's $\kappa_b$).

In [ ]:
R_C, G0_C = 10.0, 100.0
G_C = [round(g, 2) for g in np.arange(100.6, 104.21, 0.3)] + [105.0, 106.0, 108.0, 110.0, 113.0, 116.0]
def analytic_state(g, r):
    A = satellite_hb(g/g2_max, r)
    a = A["a0"] + A["a1"] + A["am1"]; b = A["b0"] + A["bm1"] + A["b2"]
    return [float(np.sqrt(2)*a.real), float(np.sqrt(2)*b.real), float(np.sqrt(2)*a.imag), float(2*np.sqrt(2)*b.imag)]
json.dump({"ratio": R_C, "g0_MHz": G0_C, "g2_MHz": G_C, "N_max_Ta": 20000, "n_traj": 40,
           "seed_g0": analytic_state(G0_C, R_C), "seed_analytic": [analytic_state(g, R_C) for g in G_C]},
          open(OUT_DIR + "torus_crisis/bubble_crisis_input_22_10_2026.json", "w"))
print(f"bubble at eps_d/g2 = {R_C}: {lower[R_C]:.2f} - {upper[R_C]:.2f} MHz")

RUN_JULIA_C = False
OUT_C = OUT_DIR + "torus_crisis/bubble_crisis_lifetime_22_10_2026.json"
if RUN_JULIA_C or not os.path.exists(OUT_C):
    run = subprocess.run(["julia", "-t", "8", "--project=/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia", HERE + "bubble_crisis_lifetime.jl"],
                         capture_output=True, text=True)
    print(run.stdout[-1500:], run.stderr[-1500:])
C = json.load(open(OUT_C))
KT_A = C["T_a_over_kappa_inv"]                          # kappa_b T_a
N_MAX = C["N_max_Ta"]
print(f"chaotic attractor at g0 = {C['g0_MHz']} MHz: lambda_1/kappa_b = {C['lambda_g0_over_kappa']:.3f}")

In [ ]:
def family(g, fam):
    return [x for x in C["runs"] if x["g2_MHz"] == g and x["family"] == fam]
def tau_mle(runs):
    # exponential MLE with censoring: total observed time / number of escapes (in 1/kappa_b)
    t = [x["t_esc_Ta"] if x["t_esc_Ta"] is not None else N_MAX for x in runs]
    n_esc = sum(x["t_esc_Ta"] is not None for x in runs)
    return (np.sum(t)*KT_A/n_esc if n_esc else np.inf), n_esc
def bhattacharyya(r1, r2, bins=40):
    z1 = np.concatenate([np.array(x["chaos_re"]) + 1j*np.array(x["chaos_im"]) for x in r1 if len(x["chaos_re"]) > 20] or [np.array([])])
    z2 = np.concatenate([np.array(x["chaos_re"]) + 1j*np.array(x["chaos_im"]) for x in r2 if len(x["chaos_re"]) > 20] or [np.array([])])
    if len(z1) < 200 or len(z2) < 200:
        return np.nan
    rng = [[min(z1.real.min(), z2.real.min()), max(z1.real.max(), z2.real.max())], [min(z1.imag.min(), z2.imag.min()), max(z1.imag.max(), z2.imag.max())]]
    h1 = np.histogram2d(z1.real, z1.imag, bins=bins, range=rng)[0]; h2 = np.histogram2d(z2.real, z2.imag, bins=bins, range=rng)[0]
    return np.sum(np.sqrt(h1/h1.sum()*h2/h2.sum()))

In [ ]:
TAB = []
print(" g2 (MHz) | attractor: tau (1/kappa_b), escaped | seed: tau, escaped | overlap of the chaotic sections | where they go (period, <|a|>)")
for g in G_C:
    ra, rs = family(g, "attractor"), family(g, "seed")
    (ta, na), (ts, ns) = tau_mle(ra), tau_mle(rs)
    ov = bhattacharyya(ra, rs)
    dest = sorted({(x["period"], round(x["abs_a_end"], 1)) for x in ra + rs if x["t_esc_Ta"] is not None})
    TAB.append((g, ta, na, ts, ns, ov))
    print(f"  {g:6.1f} | {ta:8.1f}  {na:2d}/40 | {ts:8.1f}  {ns:2d}/40 | {ov:5.2f} | {dest[:4]}")
TAB = np.array(TAB, float)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(3.2*figsize_car[0], 1.1*figsize_car[1]))
ax = axes[0]
for col, lab, c in ((1, "from the bubble's attractor", "k"), (3, "from the analytic seed", "tab:orange")):
    ok = np.isfinite(TAB[:, col])
    ax.semilogy(TAB[ok, 0], TAB[ok, col], "o-", ms=3, color=c, label=lab)
    cens = ~ok
    ax.semilogy(TAB[cens, 0], np.full(cens.sum(), N_MAX*KT_A), "^", color=c, ms=4)
ax.axvline(upper[R_C], color="gold", lw=1.5, label="end of the bubble")
ax.axhline(N_MAX*KT_A, color="gray", ls=":", lw=0.8)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\kappa_b\tau$ (escape time, code $\kappa_b$)"); ax.legend(fontsize=5)
ax.set_title(rf"$\varepsilon_d = {R_C:g}\,g_2$; triangles: no escape within the run", fontsize=6)
ax = axes[1]
for g, ls in ((101.8, "-"), (103.0, "--"), (105.0, ":")):
    for fam, c in (("attractor", "k"), ("seed", "tab:orange")):
        t = np.sort([x["t_esc_Ta"]*KT_A if x["t_esc_Ta"] is not None else np.inf for x in family(g, fam)])
        ax.semilogy(np.r_[0, t[np.isfinite(t)]], 1 - np.r_[0, np.arange(1, np.isfinite(t).sum() + 1)]/len(t) + 1e-9, ls, color=c, drawstyle="steps-post",
                    label=f"{g} MHz, {fam}")
ax.set_ylim(1/50, 1.05); ax.set_xlabel(r"$\kappa_b t$"); ax.set_ylabel("still chaotic"); ax.legend(fontsize=4.5)
ax = axes[2]
g_show = 105.0
for fam, c in (("attractor", "k"), ("seed", "tab:orange")):
    z = np.concatenate([np.array(x["chaos_re"]) + 1j*np.array(x["chaos_im"]) for x in family(g_show, fam)])
    ax.scatter(z.real, z.imag, s=0.3, color=c, alpha=0.5, lw=0, label=fam)
ax.set_aspect("equal"); ax.set_xlabel(r"Re $a(nT_a)$"); ax.set_ylabel(r"Im $a(nT_a)$"); ax.legend(fontsize=5, markerscale=10)
ax.set_title(f"chaotic part of the section at {g_show} MHz (beyond the bubble)", fontsize=6)
plt.tight_layout()
plt.savefig(FIG_DIR + "bubble_crisis_lifetime_22_10_2026.pdf", bbox_inches="tight")
plt.show()

## Conclusions
**1. End of the bubble.** The reverse period doubling of the satellite (solid gold) runs from 99.1 MHz at $\varepsilon_d/2g_2 = 6.75$ to 119.8 MHz at 2.25, and leaves the map below $\varepsilon_d/2g_2 \simeq 2.1$.
- For $\varepsilon_d/2g_2 \gtrsim 5$ it **is** the red → blue transition of the 22/10 map, within one grid cell (99.5 vs 99.1 MHz at 6.75, 103.3 vs 103.9 MHz at 5).
- For smaller cats the chaotic band stops inside the bubble (87–93 MHz for $\varepsilon_d/2g_2 = 2.25$–4), where the trajectory falls on the cat (light blue in the end-state panel). There the red → blue transition is a crisis inside the bubble, not its edge.
- Above the gold line the 22/10 map has no chaotic point at all.

**2. Chaos above the bubble in the 12/10 map comes from the initial state, not from the satellite.** On the 386 grid points above the end of the bubble ($\varepsilon_d/g_2 = 4.5$–13.5):
- From the exact satellite, with the same short transient as in 12/10 ($15/\kappa_b$): $\lambda_1 > 0$ at **0** points. The satellite is a stable attractor there.
- From the analytic seed, $\lambda_1 > 0$ at 303 points ($15/\kappa_b$; the stored map has 307), still 148 with $300/\kappa_b$. Only **1** trajectory reaches the satellite within $2000/\kappa_b$.
- The reason is the **size of the basin**: kicks from the satellite towards the analytic seed come back for 83 % of the points at $\epsilon = 0.003$, 64 % at 0.01 and **0 %** at 0.03, while the analytic seed is 0.07–0.12 away ($x^\lambda$ units). Starting "a bit off" the satellite means starting outside its basin.
- Outside it, the trajectory goes to a **coexisting chaotic set**: chaotic for less than $300/\kappa_b$ (161 points), for more (74), still chaotic at $2000/\kappa_b$ (75: a chaotic attractor or a very long-lived saddle), then onto other regular attractors that are not the satellite; 75 go there directly.
  This set is the one found by the up-sweeps of `beyond_cusp_regimes.ipynb` (chaotic bands at 111–120 and 128–134 MHz at $\varepsilon_d = 10.12\,g_2$), which coexist with the stable satellite above the bubble.

**3. The two differences between the maps.** The short transient of 12/10 counts transient chaos as chaos (303 → 148 points with $300/\kappa_b$), but the dominant effect is the initial state: from the exact satellite there is no chaos even with the short transient.

So above the bubble the satellite does not become chaotic. It is stable, with a basin of width ~0.01–0.03 along the seed direction, surrounded by a chaotic set and other periodic attractors. A map that starts near (but not on) the satellite measures that coexistence, not an instability of the satellite.

**4. Where the chaos beyond the bubble comes from ($\varepsilon_d = 10\,g_2$, section 4).**
- The bubble's chaotic attractor ($\lambda_1 = 5.6\,\kappa_b$ at 100 MHz) undergoes a **boundary crisis at $g_c \simeq 101.4$ MHz**, 2.5 MHz before the end of the bubble (103.9 MHz). Below $g_c$ no trajectory escapes within $1490/\kappa_b$.
  Above it the chaos is transient, with $\tau = A\,(g_2 - g_c)^{-\gamma}$, $\gamma \simeq 1.3$ (fit on 101.5–103.3 MHz, rms 0.07 in $\ln\tau$): from $7300/\kappa_b$ at 101.5 MHz down to about $200/\kappa_b$ at the end of the bubble.
- Beyond the bubble the lifetime stays at 150–600$/\kappa_b$ up to 116 MHz (not monotonic: longer at 106 and 113 MHz).
- Trajectories started from the analytic seed have the **same lifetime** as those started on the bubble's attractor at every $g_2$ (within the statistics of 40 runs), and their chaotic sections **overlap** (Bhattacharyya coefficient 0.98).
  So the chaos met slightly off the satellite beyond the bubble is the **bubble's chaotic attractor turned into a chaotic saddle by the crisis**, not a new chaotic set.
- The trajectories leave it for other periodic orbits: a period-2 orbit ($|a(nT_a)| = 3.0$ and 3.9, probably the $\omega_a/2$ orbit of `Hill_method_cat_fold`) up to about 106 MHz, then period-5 and large period-1 orbits. Never for the satellite, whose basin is too thin.
- Same scenario as at $\varepsilon_d = 5.2\,g_2$ (`transient_chaos_shannon_entropy.ipynb`: crisis at 87.6 MHz, $\gamma \simeq 1.7$), but there the crisis is far inside the bubble (80.5–116.4 MHz).
  The gap between the crisis and the end of the bubble grows as the cat gets smaller, which is why the red band of the 22/10 map ends inside the bubble for small cats and at its edge for large ones.
- **Not explained by this saddle:** the 75 points of section 3 still chaotic after $2000/\kappa_b$. 59 of them lie at 115–120 MHz (116.4–119.9 MHz at $\varepsilon_d = 10\,g_2$): this is the separate chaotic band of the up-sweeps of `beyond_cusp_regimes` (116–120 MHz), above the range of section 4.
  10 lie at 100–105 MHz for $\varepsilon_d/g_2 = 12.5$–13.5, just beyond an end of the bubble that is close to the crisis there, where the saddle lives long.
